# HR Assistant Agent — Ground Truth Evaluations

This notebook demonstrates evaluation of an agentic application with ground truth using Amazon Bedrock AgentCore Evaluations:

| Interface | When to use |
|---|---|
| **EvaluationClient** | You already have agent sessions in CloudWatch. Evaluate specific sessions against reference inputs. |
| **OnDemandEvaluationDatasetRunner** | You have a test dataset. You want to invoke the agent for every scenario and evaluate the results. |

### The HR Assistant Agent

We'll deploy an **HR Assistant** for Acme Corp — a Strands agent that helps employees with:
- PTO balance checks and time-off requests
- HR policy lookups (PTO, remote work, parental leave)
- Benefits information (health, dental, vision, 401k)
- Pay stub retrieval

### What You'll Learn
- How to use `EvaluationClient` to evaluate existing agent sessions logged in Amazon CloudWatch with ground-truth references
- How to use `OnDemandEvaluationDatasetRunner` to run automated dataset evaluations
- How to interpret evaluation results across built-in evaluators (Correctness, GoalSuccessRate, Trajectory)

### Tutorial Details

| Information | Details |
|---|---|
| Agent framework | Strands Agents |
| Runtime | Amazon Bedrock AgentCore Runtime |
| Evaluation SDK | `bedrock-agentcore` |
| AWS services | AgentCore Runtime, AgentCore Evaluations, CloudWatch Logs |

### Prerequisites
- Python 3.10+
- AWS credentials with permissions for AgentCore, Lambda, CloudWatch, ECR, IAM
- Docker running locally (for agent container image build)

## Step 1: Install Dependencies

In [13]:
!pip install -r requirements.txt -q


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


## Step 2: Configuration

Import libraries and configure your AWS session.

In [14]:
import boto3
import json
import time
import uuid
from datetime import timedelta
from boto3.session import Session
from IPython.display import display, Markdown

region = "us-east-1"
boto_session = Session(region_name=region)
REGION = boto_session.region_name

print(f"Region  : {REGION}")

Region  : us-east-1


## Step 3: Deploy the HR Assistant Agent

We deploy the HR Assistant to **AgentCore Runtime** using the AWS SDK directly (no starter toolkit).
The deployment steps are:

1. **ECR authentication** — use `boto3` to get a temporary ECR token and run `docker login`
2. **Docker build** — build a container image from `hr_assistant_agent.py` and the local `Dockerfile`
3. **Docker push** — push the image to Amazon ECR
4. **CreateAgentRuntime / UpdateAgentRuntime** — register or update the agent endpoint via `bedrock-agentcore-control`

If a `.bedrock_agentcore.yaml` from a previous run is present, its ECR repository, IAM role, and
existing agent ID are reused so the cell is idempotent (re-running triggers an update, not a duplicate create).

In [3]:
%%writefile hr_assistant_agent.py
"""
HR Assistant Agent — Strands agent deployed on Bedrock AgentCore Runtime.

Tools (deterministic / mock data for reproducible evaluations):
  get_pto_balance        — remaining PTO days for an employee
  submit_pto_request     — request time off
  lookup_hr_policy       — company policy documents
  get_benefits_summary   — health, dental, vision, 401k, life insurance details
  get_pay_stub           — pay stub for a given period
"""

import logging
import re

from bedrock_agentcore.runtime import BedrockAgentCoreApp
from strands import Agent, tool
from strands.models import BedrockModel

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

app = BedrockAgentCoreApp()

# ---------------------------------------------------------------------------
# Mock data
# ---------------------------------------------------------------------------

_PTO_BALANCES = {
    "EMP-001": {"total_days": 15, "used_days": 5,  "remaining_days": 10},
    "EMP-002": {"total_days": 15, "used_days": 12, "remaining_days": 3},
    "EMP-042": {"total_days": 20, "used_days": 7,  "remaining_days": 13},
}

_HR_POLICIES = {
    "pto": (
        "PTO Policy: Full-time employees accrue 15 days of PTO per year (20 days after 3 years). "
        "PTO requests must be submitted at least 2 business days in advance. "
        "Unused PTO up to 5 days rolls over to the next year. "
        "PTO cannot be taken in advance of accrual."
    ),
    "remote_work": (
        "Remote Work Policy: Employees may work remotely up to 3 days per week with manager approval. "
        "Core collaboration hours are 10am-3pm local time. "
        "A dedicated workspace with reliable internet (25 Mbps+) is required. "
        "Employees must be reachable via Slack and email during core hours."
    ),
    "parental_leave": (
        "Parental Leave Policy: Primary caregivers receive 16 weeks of fully paid parental leave. "
        "Secondary caregivers receive 6 weeks of fully paid parental leave. "
        "Leave may begin up to 2 weeks before the expected birth or adoption date. "
        "Benefits continue unchanged during parental leave."
    ),
    "code_of_conduct": (
        "Code of Conduct: All employees are expected to treat colleagues, customers, and partners "
        "with respect and professionalism. Harassment, discrimination, and retaliation of any kind "
        "are strictly prohibited. Violations should be reported to HR or via the anonymous hotline."
    ),
}

_BENEFITS = {
    "health": (
        "Health Insurance: The company covers 90% of premiums for employee-only coverage and 75% "
        "for family coverage. Plans available: Blue Shield PPO, Kaiser HMO, and HDHP with HSA. "
        "Annual deductible: $500 (PPO), $0 (HMO), $1,500 (HDHP). "
        "Open enrollment is each November for the following calendar year."
    ),
    "dental": (
        "Dental Insurance: 100% coverage for preventive care (cleanings, X-rays). "
        "80% coverage for basic restorative care (fillings, extractions). "
        "50% coverage for major restorative care (crowns, bridges). "
        "Annual maximum benefit: $2,000 per person. Orthodontia lifetime maximum: $1,500."
    ),
    "vision": (
        "Vision Insurance: Annual eye exam covered in full. "
        "Frames or contacts allowance: $200 per year. "
        "Laser vision correction discount: 15% off at participating providers."
    ),
    "401k": (
        "401(k) Plan: The company matches 100% of employee contributions up to 4% of salary. "
        "An additional 50% match on the next 2% (total effective match up to 5%). "
        "Employees are eligible to contribute immediately; company match vests over 3 years. "
        "2026 IRS contribution limit: $23,500 (under 50), $31,000 (age 50+)."
    ),
    "life_insurance": (
        "Life Insurance: Basic life insurance of 2x annual salary provided at no cost. "
        "Employees may purchase supplemental coverage up to 5x salary during open enrollment. "
        "Accidental death and dismemberment (AD&D) coverage equal to basic life benefit is included."
    ),
}

_PAY_STUBS = {
    ("EMP-001", "2025-12"): {
        "gross_pay": 8333.33, "federal_tax": 1458.33, "state_tax": 416.67,
        "social_security": 516.67, "medicare": 120.83, "health_premium": 125.00,
        "401k_contribution": 333.33, "net_pay": 5362.50, "period": "December 2025",
    },
    ("EMP-001", "2026-01"): {
        "gross_pay": 8333.33, "federal_tax": 1458.33, "state_tax": 416.67,
        "social_security": 516.67, "medicare": 120.83, "health_premium": 125.00,
        "401k_contribution": 333.33, "net_pay": 5362.50, "period": "January 2026",
    },
    ("EMP-042", "2026-01"): {
        "gross_pay": 10416.67, "federal_tax": 1875.00, "state_tax": 520.83,
        "social_security": 645.83, "medicare": 151.04, "health_premium": 200.00,
        "401k_contribution": 416.67, "net_pay": 6607.30, "period": "January 2026",
    },
}

_PTO_REQUEST_COUNTER = {"n": 0}


# ---------------------------------------------------------------------------
# Strands tools
# ---------------------------------------------------------------------------

@tool
def get_pto_balance(employee_id: str) -> dict:
    """
    Return the current PTO balance for an employee.

    Args:
        employee_id: Employee identifier (e.g. EMP-001)

    Returns:
        Dict with total_days, used_days, and remaining_days.
    """
    balance = _PTO_BALANCES.get(employee_id)
    if balance:
        return {"employee_id": employee_id, **balance}
    return {"employee_id": employee_id, "error": f"Employee {employee_id} not found."}


@tool
def submit_pto_request(
    employee_id: str,
    start_date: str,
    end_date: str,
    reason: str = "Personal time off",
) -> dict:
    """
    Submit a PTO request for an employee.

    Args:
        employee_id: Employee identifier (e.g. EMP-001)
        start_date:  First day of leave in YYYY-MM-DD format
        end_date:    Last day of leave in YYYY-MM-DD format
        reason:      Optional reason for the request

    Returns:
        Dict with request_id, status, and confirmation message.
    """
    _PTO_REQUEST_COUNTER["n"] += 1
    request_id = f"PTO-2026-{_PTO_REQUEST_COUNTER['n']:03d}"
    return {
        "request_id": request_id,
        "employee_id": employee_id,
        "start_date": start_date,
        "end_date": end_date,
        "reason": reason,
        "status": "APPROVED",
        "message": f"PTO request {request_id} approved for {employee_id} from {start_date} to {end_date}.",
    }


@tool
def lookup_hr_policy(topic: str) -> dict:
    """
    Look up a company HR policy document by topic.

    Args:
        topic: Policy topic. Supported values: pto, remote_work, parental_leave, code_of_conduct

    Returns:
        Dict with topic and policy_text.
    """
    key = topic.lower().replace(" ", "_").replace("-", "_")
    text = _HR_POLICIES.get(key)
    if text:
        return {"topic": topic, "policy_text": text}
    return {"topic": topic, "error": f"Policy '{topic}' not found. Available: {list(_HR_POLICIES.keys())}"}


@tool
def get_benefits_summary(benefit_type: str) -> dict:
    """
    Return a summary of a specific employee benefit.

    Args:
        benefit_type: Type of benefit. Supported values: health, dental, vision, 401k, life_insurance

    Returns:
        Dict with benefit_type and summary text.
    """
    key = benefit_type.lower().replace(" ", "_").replace("-", "_")
    text = _BENEFITS.get(key)
    if text:
        return {"benefit_type": benefit_type, "summary": text}
    return {"benefit_type": benefit_type, "error": f"Benefit '{benefit_type}' not found. Available: {list(_BENEFITS.keys())}"}


@tool
def get_pay_stub(employee_id: str, period: str) -> dict:
    """
    Retrieve a pay stub for an employee for a specific pay period.

    Args:
        employee_id: Employee identifier (e.g. EMP-001)
        period:      Pay period in YYYY-MM format (e.g. 2026-01)

    Returns:
        Dict with gross pay, deductions, and net pay.
    """
    stub = _PAY_STUBS.get((employee_id, period))
    if stub:
        return {"employee_id": employee_id, **stub}
    return {"employee_id": employee_id, "period": period, "error": f"Pay stub not found for {employee_id} period {period}."}


# ---------------------------------------------------------------------------
# Agent
# ---------------------------------------------------------------------------

SYSTEM_PROMPT = """You are a helpful HR Assistant for Acme Corp.

You help employees with:
- Checking PTO (paid time off) balances
- Submitting PTO requests
- Looking up HR policies (PTO, remote work, parental leave, code of conduct)
- Understanding employee benefits (health, dental, vision, 401k, life insurance)
- Retrieving pay stub information

Always use the available tools to answer questions accurately. Do not make up
policy details, benefit amounts, or pay information — look them up.
Be concise, professional, and friendly."""

_MODEL = BedrockModel(model_id="us.amazon.nova-lite-v1:0")
_TOOLS = [get_pto_balance, submit_pto_request, lookup_hr_policy, get_benefits_summary, get_pay_stub]

# Session cache: session_id -> Agent (preserves conversation history across turns)
_SESSION_AGENTS: dict[str, Agent] = {}


@app.entrypoint
async def invoke(payload, context):
    """Handle an agent invocation from AgentCore Runtime."""
    prompt = payload.get("prompt", "")
    session_id = context.session_id
    logger.info("Received prompt (session=%s): %s", session_id, prompt[:80])

    if session_id and session_id in _SESSION_AGENTS:
        agent = _SESSION_AGENTS[session_id]
    else:
        agent = Agent(model=_MODEL, tools=_TOOLS, system_prompt=SYSTEM_PROMPT)
        if session_id:
            _SESSION_AGENTS[session_id] = agent

    parts = []
    async for event in agent.stream_async(prompt):
        if "data" in event:
            parts.append(str(event["data"]))
    response = "".join(parts)
    # Strip inline <thinking>...</thinking> blocks so spans contain only the final answer
    response = re.sub(r"<thinking>.*?</thinking>", "", response, flags=re.DOTALL).strip()
    return response


if __name__ == "__main__":
    app.run()

Overwriting hr_assistant_agent.py


In [4]:
from bedrock_agentcore_starter_toolkit import Runtime

_REGION = REGION or "us-east-1"

agentcore_runtime = Runtime()
agentcore_runtime.configure(
    entrypoint="hr_assistant_agent.py",
    agent_name="hr_assistant_eval_tutorial",
    region=_REGION,
    auto_create_execution_role=True,
    auto_create_ecr=True,
    requirements_file="requirements.txt",
    non_interactive=True,
)
print("Configuration complete.")

print("\nDeploying HR Assistant Agent ...")
print("  This takes ~5 minutes on first run (image build + push + runtime creation).")
print()

_launch = agentcore_runtime.launch(auto_update_on_conflict=True)

print(f"\nLaunch complete.")
print(f"  agent_id  : {_launch.agent_id}")
print(f"  agent_arn : {_launch.agent_arn}")

Entrypoint parsed: file=/Users/bhrsrini/projects/Agentcore/Policy/amazon-bedrock-agentcore-samples/01-tutorials/07-AgentCore-evaluations/05-groundtruth-based-evalautions/hr_assistant_agent.py, bedrock_agentcore_name=hr_assistant_agent


Memory disabled - agent will be stateless


Configuring BedrockAgentCore agent: hr_assistant_eval_tutorial


Memory disabled


Network mode: PUBLIC


...

   /aws/bedrock-agentcore/runtimes/hr_assistant_eval_tutorial-xfZ3yiH356-DEFAULT --log-stream-name-prefix "2026/03/31/\[runtime-logs]"


   /aws/bedrock-agentcore/runtimes/hr_assistant_eval_tutorial-xfZ3yiH356-DEFAULT --log-stream-names "otel-rt-logs"


💡 Tail logs with: aws logs tail /aws/bedrock-agentcore/runtimes/hr_assistant_eval_tutorial-xfZ3yiH356-DEFAULT --log-stream-name-prefix "2026/03/31/\[runtime-logs]" --follow


💡 Or view recent logs: aws logs tail /aws/bedrock-agentcore/runtimes/hr_assistant_eval_tutorial-xfZ3yiH356-DEFAULT --log-stream-name-prefix "2026/03/31/\[runtime-logs]" --since 1h



Launch complete.
  agent_id  : hr_assistant_eval_tutorial-xfZ3yiH356
  agent_arn : arn:aws:bedrock-agentcore:us-east-1:849138760372:runtime/hr_assistant_eval_tutorial-xfZ3yiH356


In [5]:
import time

print("Waiting for agent to reach READY status ...")

_POLL_INTERVAL = 15   # seconds between status checks
_MAX_WAIT      = 600  # 10-minute timeout

_elapsed = 0
while _elapsed < _MAX_WAIT:
    _status_result = agentcore_runtime.status()
    _agent_info    = _status_result.agent or {}
    _agent_status  = _agent_info.get("status", "UNKNOWN")
    print(f"  [{_elapsed:>3}s] status = {_agent_status}")

    if _agent_status in ("READY", "ACTIVE"):
        print(f"\nAgent is {_agent_status}. Proceeding.")
        break
    if _agent_status in ("FAILED", "CREATE_FAILED", "UPDATE_FAILED"):
        raise RuntimeError(
            f"Agent deployment failed with status '{_agent_status}'.\n"
            f"Details: {_agent_info}"
        )

    time.sleep(_POLL_INTERVAL)
    _elapsed += _POLL_INTERVAL
else:
    raise TimeoutError(
        f"Agent did not reach READY status within {_MAX_WAIT}s. "
        "Check the AgentCore console for details."
    )

Waiting for agent to reach READY status ...


Retrieved Bedrock AgentCore status for: hr_assistant_eval_tutorial


  [  0s] status = READY

Agent is READY. Proceeding.


In [6]:
AGENT_ID     = _launch.agent_id
AGENT_ARN    = _launch.agent_arn
CW_LOG_GROUP = f"/aws/bedrock-agentcore/runtimes/{AGENT_ID}-DEFAULT"

agentcore_client = boto3.client("bedrock-agentcore", region_name=_REGION)

print(f"AGENT_ID     : {AGENT_ID}")
print(f"AGENT_ARN    : {AGENT_ARN}")
print(f"CW_LOG_GROUP : {CW_LOG_GROUP}")

AGENT_ID     : hr_assistant_eval_tutorial-xfZ3yiH356
AGENT_ARN    : arn:aws:bedrock-agentcore:us-east-1:849138760372:runtime/hr_assistant_eval_tutorial-xfZ3yiH356
CW_LOG_GROUP : /aws/bedrock-agentcore/runtimes/hr_assistant_eval_tutorial-xfZ3yiH356-DEFAULT


In [7]:
# Persist agent info for the programmatic_evaluators notebook
%store AGENT_ID
%store AGENT_ARN
%store CW_LOG_GROUP
%store REGION

Stored 'AGENT_ID' (str)
Stored 'AGENT_ARN' (str)
Stored 'CW_LOG_GROUP' (str)
Stored 'REGION' (str)


## Step 3b: Create Custom (LLM-as-a-Judge) Evaluators

In addition to built-in evaluators, you can define your own evaluation criteria using
**LLM-as-a-judge custom evaluators**. These accept natural language instructions that
can reference **ground truth placeholders** automatically substituted at evaluation time.

### Ground truth placeholders

| Level | Available placeholders |
|---|---|
| **TRACE** | `{context}`, `{assistant_turn}`, `{expected_response}` |
| **SESSION** | `{context}`, `{available_tools}`, `{actual_tool_trajectory}`, `{expected_tool_trajectory}`, `{assertions}` |

For example, a trace-level evaluator comparing response similarity would include
`{assistant_turn}` and `{expected_response}` in its instructions. When the evaluator runs,
the service substitutes those placeholders with the actual agent output and the
`expectedResponse` from `ReferenceInputs`.

### What we'll create

| Evaluator | Level | Placeholders | Description |
|---|---|---|---|
| `HRResponseSimilarity` | TRACE | `{assistant_turn}`, `{expected_response}` | How closely the agent's response matches the expected answer |
| `HRAssertionChecker` | SESSION | `{actual_tool_trajectory}`, `{expected_tool_trajectory}`, `{assertions}` | Whether the agent called the right tools and satisfied all session assertions |

In [8]:
import uuid

_SUFFIX = uuid.uuid4().hex[:8]
_cp = boto3.client("bedrock-agentcore-control", region_name=_REGION)

# ---------------------------------------------------------------------------
# Trace-level: HRResponseSimilarity
# Compares the agent's response to the expected_response reference input.
# {assistant_turn} → actual agent output
# {expected_response} → expectedResponse field in ReferenceInputs
# ---------------------------------------------------------------------------
print("Creating HRResponseSimilarity (TRACE) ...")
_resp_sim = _cp.create_evaluator(
    evaluatorName=f"HRResponseSimilarity_{_SUFFIX}",
    level="TRACE",
    evaluatorConfig={
        "llmAsAJudge": {
            "instructions": (
                "Compare the agent's response with the expected response.\n"
                "Agent response: {assistant_turn}\n"
                "Expected response: {expected_response}\n\n"
                "Rate how closely the agent's response matches the expected response. "
                "Focus on whether the key facts, numbers, and conclusions agree."
            ),
            "ratingScale": {
                "numerical": [
                    {
                        "value": 0.0,
                        "label": "not_similar",
                        "definition": "Response is factually different or missing key information from the expected response.",
                    },
                    {
                        "value": 0.5,
                        "label": "partially_similar",
                        "definition": "Response captures some expected content but omits or misrepresents parts.",
                    },
                    {
                        "value": 1.0,
                        "label": "highly_similar",
                        "definition": "Response is semantically equivalent to the expected response — all key facts match.",
                    },
                ]
            },
            "modelConfig": {
                "bedrockEvaluatorModelConfig": {
                    "modelId": "us.amazon.nova-lite-v1:0",
                    "inferenceConfig": {"maxTokens": 512},
                }
            },
        }
    },
)
CUSTOM_RESPONSE_SIMILARITY_ID = _resp_sim["evaluatorId"]
print(f"  evaluatorId : {CUSTOM_RESPONSE_SIMILARITY_ID}")

# ---------------------------------------------------------------------------
# Session-level: HRAssertionChecker
# Evaluates tool trajectory compliance and assertion satisfaction.
# {actual_tool_trajectory}   → tools the agent actually called
# {expected_tool_trajectory} → expectedTrajectory from ReferenceInputs
# {assertions}               → assertions list from ReferenceInputs
# ---------------------------------------------------------------------------
print("\nCreating HRAssertionChecker (SESSION) ...")
_assert_chk = _cp.create_evaluator(
    evaluatorName=f"HRAssertionChecker_{_SUFFIX}",
    level="SESSION",
    evaluatorConfig={
        "llmAsAJudge": {
            "instructions": (
                "Evaluate whether the agent fulfilled the session requirements.\n\n"
                "Expected tool trajectory: {expected_tool_trajectory}\n"
                "Actual tool trajectory: {actual_tool_trajectory}\n"
                "Assertions to verify: {assertions}\n\n"
                "Score the agent on how well it followed the expected tool trajectory "
                "and satisfied every listed assertion."
            ),
            "ratingScale": {
                "numerical": [
                    {
                        "value": 0.0,
                        "label": "failed",
                        "definition": "Agent did not follow the trajectory and failed most assertions.",
                    },
                    {
                        "value": 0.5,
                        "label": "partial",
                        "definition": "Agent partially followed the trajectory or satisfied only some assertions.",
                    },
                    {
                        "value": 1.0,
                        "label": "passed",
                        "definition": "Agent followed the expected trajectory and satisfied all assertions.",
                    },
                ]
            },
            "modelConfig": {
                "bedrockEvaluatorModelConfig": {
                    "modelId": "us.amazon.nova-lite-v1:0",
                    "inferenceConfig": {"maxTokens": 512},
                }
            },
        }
    },
)
CUSTOM_ASSERTION_CHECKER_ID = _assert_chk["evaluatorId"]
print(f"  evaluatorId : {CUSTOM_ASSERTION_CHECKER_ID}")

print(f"\nCustom evaluators ready:")
print(f"  HRResponseSimilarity (TRACE)   : {CUSTOM_RESPONSE_SIMILARITY_ID}")
print(f"  HRAssertionChecker   (SESSION) : {CUSTOM_ASSERTION_CHECKER_ID}")

Creating HRResponseSimilarity (TRACE) ...


  evaluatorId : HRResponseSimilarity_a007e092-v0ojZ8ARHR

Creating HRAssertionChecker (SESSION) ...


  evaluatorId : HRAssertionChecker_a007e092-zUFaMBBhck

Custom evaluators ready:
  HRResponseSimilarity (TRACE)   : HRResponseSimilarity_a007e092-v0ojZ8ARHR
  HRAssertionChecker   (SESSION) : HRAssertionChecker_a007e092-zUFaMBBhck


## Step 4: Invoke the Agent to Generate Sessions

Before we can evaluate, we need agent sessions with CloudWatch spans. We'll invoke the agent
for several scenarios and record the session IDs for use with `EvaluationClient`.

Each session corresponds to one evaluation scenario.

In [9]:
def invoke_agent(prompt: str, session_id: str) -> str:
    """Send a single prompt to the HR assistant and return its text response."""
    resp = agentcore_client.invoke_agent_runtime(
        agentRuntimeArn=AGENT_ARN,
        qualifier="DEFAULT",
        runtimeSessionId=session_id,
        payload=json.dumps({"prompt": prompt}).encode("utf-8"),
    )
    raw = resp["response"].read().decode("utf-8")
    parts = []
    for line in raw.splitlines():
        if line.startswith("data: "):
            chunk = line[len("data: "):]
            try:
                chunk = json.loads(chunk)
            except Exception:
                pass
            parts.append(str(chunk))
    return "".join(parts) if parts else raw


def run_session(turns: list[str], session_prefix: str) -> str:
    """Invoke a multi-turn session and return its session ID."""
    session_id = f"{session_prefix}-{uuid.uuid4()}"
    print(f"Session: {session_id}")
    for turn_input in turns:
        print(f"  > {turn_input[:70]}")
        response = invoke_agent(turn_input, session_id)
        print(f"  < {response[:100]}")
    return session_id

In [10]:
# --- Single-turn sessions ---

print("=== Single-Turn Sessions ===")

session_pto_balance = run_session(
    ["What is the current PTO balance for employee EMP-001?"],
    "pto-balance-check"
)

session_submit_pto = run_session(
    ["Please submit a PTO request for employee EMP-001 from 2026-04-14 to 2026-04-16 for a family vacation."],
    "submit-pto-request"
)

session_pay_stub = run_session(
    ["Can you pull up the January 2026 pay stub for employee EMP-001?"],
    "pay-stub-lookup"
)

print("\nSingle-turn sessions created.")

=== Single-Turn Sessions ===
Session: pto-balance-check-6174aaad-8137-428b-8b54-64aa650cbedf
  > What is the current PTO balance for employee EMP-001?


  < "The current PTO balance for employee EMP-001 is 10 remaining days. They have a total of 15 days all
Session: submit-pto-request-3b5a7027-eedb-427b-ab71-357f5ea55393
  > Please submit a PTO request for employee EMP-001 from 2026-04-14 to 20


  < "Hi there! Your PTO request from 2026-04-14 to 2026-04-16 for a family vacation has been successfull
Session: pay-stub-lookup-739b4301-7fed-4486-922f-fae9b33cca3a
  > Can you pull up the January 2026 pay stub for employee EMP-001?


  < "Here is the pay stub for employee EMP-001 for January 2026:\n\n- Gross Pay: $8,333.33\n- Federal Ta

Single-turn sessions created.


In [11]:
# --- Multi-turn session: PTO planning ---

print("=== Multi-Turn Session: PTO Planning ===")

session_pto_planning = run_session(
    [
        "How many PTO days do I have left? My employee ID is EMP-001.",
        "Great. I'd like to take December 23 to December 25 off. Please submit a request.",
        "Remind me — what is the policy on rolling over unused PTO?",
    ],
    "pto-planning-session"
)

print("\nMulti-turn session created.")

=== Multi-Turn Session: PTO Planning ===
Session: pto-planning-session-4d42b1e2-b0c3-43c5-9298-05ead1cf4822
  > How many PTO days do I have left? My employee ID is EMP-001.


  < "Hi there! Based on your employee ID, you have 10 PTO days remaining. You've used 5 days out of your
  > Great. I'd like to take December 23 to December 25 off. Please submit 


  < "Your PTO request for December 23 to December 25 has been approved. Your request ID is PTO-2026-001.
  > Remind me — what is the policy on rolling over unused PTO?


  < "According to Acme Corp's PTO policy, unused PTO up to 5 days rolls over to the next year. This mean

Multi-turn session created.


In [12]:
# --- Multi-turn session: New employee onboarding ---

print("=== Multi-Turn Session: New Employee Onboarding ===")

session_onboarding = run_session(
    [
        "I just joined the company. What is the remote work policy?",
        "How much PTO do I get as a new employee?",
        "What life insurance benefit does the company provide?",
        "Can you check the current PTO balance for employee EMP-042?",
    ],
    "new-employee-onboarding"
)

print("\nAll sessions created. Waiting 60s for CloudWatch log ingestion...")
time.sleep(60)
print("Ready to evaluate.")

=== Multi-Turn Session: New Employee Onboarding ===
Session: new-employee-onboarding-1b08d214-9233-4050-a550-095962aa53f3
  > I just joined the company. What is the remote work policy?


  < "Here is the remote work policy for Acme Corp:\n\n**Remote Work Policy**\n- Employees may work remot
  > How much PTO do I get as a new employee?


  < "It appears there was an error because the employee ID \"EMP-NEW\" does not exist in the system. Cou
  > What life insurance benefit does the company provide?


  < "Here is the summary of the life insurance benefit provided by Acme Corp:\n\n**Life Insurance**\n- B
  > Can you check the current PTO balance for employee EMP-042?


  < "The current PTO balance for employee EMP-042 is:\n\n- **Total days**: 20\n- **Used days**: 7\n- **R

All sessions created. Waiting 60s for CloudWatch log ingestion...


Ready to evaluate.


## Step 5: EvaluationClient — Evaluate Existing Sessions

`EvaluationClient` is the right tool when you **already have agent sessions** recorded in CloudWatch.
It looks up the agent's spans for a given `session_id` and runs evaluators against them.
No agent re-invocation occurs.

### Ground-Truth Reference Inputs

`ReferenceInputs` lets you supply optional ground truth:

| Field | Evaluators that use it | Description |
|---|---|---|
| `expected_response` | `Builtin.Correctness` | The ideal response text |
| `expected_trajectory` | `Builtin.TrajectoryExactOrderMatch`, `Builtin.TrajectoryInOrderMatch`, `Builtin.TrajectoryAnyOrderMatch` | Ordered list of tool names |
| `assertions` | `Builtin.GoalSuccessRate` | Free-text assertions the session should satisfy |

Evaluators that don't need ground truth (`Helpfulness`, `ResponseRelevance`) can be included in the same call.
Each evaluator only reads the fields it needs.

In [13]:
from bedrock_agentcore.evaluation import EvaluationClient, ReferenceInputs

eval_client = EvaluationClient(region_name=REGION)

# Seed the evaluator level cache with custom evaluator IDs so the client
# doesn't need to call get_evaluator for them (BUG-001 workaround).
eval_client._evaluator_level_cache.update({
    CUSTOM_RESPONSE_SIMILARITY_ID: "TRACE",
    CUSTOM_ASSERTION_CHECKER_ID: "SESSION",
})

print(f"EvaluationClient initialised (region={REGION})")
print(f"  {CUSTOM_RESPONSE_SIMILARITY_ID} → TRACE  (custom: HRResponseSimilarity)")
print(f"  {CUSTOM_ASSERTION_CHECKER_ID} → SESSION (custom: HRAssertionChecker)")

EvaluationClient initialised (region=us-east-1)
  HRResponseSimilarity_a007e092-v0ojZ8ARHR → TRACE  (custom: HRResponseSimilarity)
  HRAssertionChecker_a007e092-zUFaMBBhck → SESSION (custom: HRAssertionChecker)


In [14]:
def display_eval_results(label: str, results: list) -> None:
    """Pretty-print EvaluationClient results as a markdown table."""
    rows = ["| Evaluator | Value | Label | Explanation |",
            "|---|---|---|---|"]
    for r in results:
        evaluator = r.get("evaluatorId", "")[:40]
        value = str(r.get("value", r.get("score", "N/A")))
        lbl = str(r.get("label", r.get("rating", "")))
        explanation = (r.get("explanation", r.get("reason", "")) or "")[:120].replace("\n", " ")
        error_code = r.get("errorCode")
        if error_code:
            lbl = f"ERR:{error_code}"
            explanation = (r.get("errorMessage", "") or "")[:120]
        rows.append(f"| `{evaluator}` | {value} | {lbl} | {explanation} |")

    if len(rows) == 2:  # only header rows, no data
        rows.append("| No results — session may be too recent or spans not yet visible | | | |")

    md = f"### {label}\n\n" + "\n".join(rows)
    display(Markdown(md))

### 5a. Single-Turn: PTO Balance — Correctness + Helpfulness + Custom ResponseSimilarity

We evaluate the PTO balance response against a known expected answer using `Builtin.Correctness`
and the custom `HRResponseSimilarity` evaluator (which uses the `{assistant_turn}` and
`{expected_response}` placeholders). Both measure factual accuracy but use different scoring rubrics.

In [15]:
pto_balance_results = eval_client.run(
    evaluator_ids=[
        "Builtin.Correctness",           # TRACE: built-in factual accuracy
        "Builtin.Helpfulness",           # TRACE: no ground truth needed
        "Builtin.ResponseRelevance",     # TRACE: no ground truth needed
        CUSTOM_RESPONSE_SIMILARITY_ID,   # TRACE: custom — uses {assistant_turn} + {expected_response}
    ],
    session_id=session_pto_balance,
    agent_id=AGENT_ID,
    look_back_time=timedelta(hours=2),
    reference_inputs=ReferenceInputs(
        expected_response="Employee EMP-001 has 10 remaining PTO days out of 15 total (5 days used).",
    ),
)

display_eval_results("PTO Balance — Correctness + Quality + Custom ResponseSimilarity", pto_balance_results)

### PTO Balance — Correctness + Quality + Custom ResponseSimilarity

| Evaluator | Value | Label | Explanation |
|---|---|---|---|
| `Builtin.Correctness` | 1.0 | Correct | The agent response provides all the core factual information present in the expected response: EMP-001 has 10 remaining  |
| `Builtin.Helpfulness` | 0.83 | Very Helpful | The user's goal is clear and straightforward: to obtain the current PTO balance for employee EMP-001. The assistant's re |
| `Builtin.ResponseRelevance` | 1.0 | Completely Yes | The user asked for the current PTO balance for employee EMP-001. The tool output provided detailed information showing:  |
| `HRResponseSimilarity_a007e092-v0ojZ8ARHR` | 1.0 | highly_similar | The agent's response and the expected response convey the same key facts: Employee EMP-001 has 10 remaining PTO days out |

### 5b. Single-Turn: PTO Submission — Assertions + Trajectory + Custom AssertionChecker

This cell runs both built-in trajectory evaluators **and** the custom `HRAssertionChecker`
(which uses `{actual_tool_trajectory}`, `{expected_tool_trajectory}`, and `{assertions}` placeholders)
plus the custom `HRResponseSimilarity` for the response. This lets you compare built-in vs. custom
scoring side by side.

In [16]:
submit_pto_results = eval_client.run(
    evaluator_ids=[
        "Builtin.GoalSuccessRate",           # SESSION: built-in assertion evaluator
        "Builtin.TrajectoryExactOrderMatch", # SESSION: built-in trajectory evaluator
        "Builtin.TrajectoryAnyOrderMatch",   # SESSION: built-in trajectory evaluator
        "Builtin.Correctness",               # TRACE: built-in response accuracy
        CUSTOM_RESPONSE_SIMILARITY_ID,       # TRACE (custom): {assistant_turn} + {expected_response}
    ],
    session_id=session_submit_pto,
    agent_id=AGENT_ID,
    look_back_time=timedelta(hours=2),
    reference_inputs=ReferenceInputs(
        expected_trajectory=["submit_pto_request"],
        assertions=[
            "Agent called submit_pto_request for employee EMP-001",
            "Agent confirmed the PTO request was approved",
            "Agent provided a request ID (e.g. PTO-2026-001)",
        ],
        expected_response="PTO request submitted and approved for EMP-001 from 2026-04-14 to 2026-04-16.",
    ),
)

display_eval_results("PTO Submission — Built-in + Custom ResponseSimilarity", submit_pto_results)

### PTO Submission — Built-in + Custom ResponseSimilarity

| Evaluator | Value | Label | Explanation |
|---|---|---|---|
| `Builtin.GoalSuccessRate` | 1.0 | Yes | The agent successfully completed all three success assertions: 1) The tool execution history shows the agent called `sub |
| `Builtin.TrajectoryExactOrderMatch` | 1.0 | Yes | Exact match: Actual trajectory ['submit_pto_request'] matches expected trajectory ['submit_pto_request'] |
| `Builtin.TrajectoryAnyOrderMatch` | 1.0 | Yes | Any-order match: All expected tools ['submit_pto_request'] found in actual ['submit_pto_request'] |
| `Builtin.Correctness` | 1.0 | Correct | The agent response correctly conveys all the core factual information from the expected response: (1) PTO request was su |
| `HRResponseSimilarity_a007e092-v0ojZ8ARHR` | 1.0 | highly_similar | The agent's response includes all key facts from the expected response: the employee ID (EMP-001), the start and end dat |

### 5c. Single-Turn: Pay Stub — Factual Correctness

Factual data retrieval scenarios are well-suited for `Builtin.Correctness` combined with
`Builtin.GoalSuccessRate`. The expected_response provides the ground truth figures.

In [17]:
pay_stub_results = eval_client.run(
    evaluator_ids=[
        "Builtin.Correctness",
        "Builtin.GoalSuccessRate",
    ],
    session_id=session_pay_stub,
    agent_id=AGENT_ID,
    look_back_time=timedelta(hours=2),
    reference_inputs=ReferenceInputs(
        expected_response="EMP-001 January 2026: gross pay $8,333.33, net pay $5,362.50.",
        assertions=[
            "Agent called get_pay_stub for EMP-001 period 2026-01",
            "Agent reported the correct gross pay of $8,333.33",
            "Agent reported the correct net pay of $5,362.50",
        ],
    ),
)

display_eval_results("Pay Stub Lookup — Correctness + GoalSuccessRate", pay_stub_results)

### Pay Stub Lookup — Correctness + GoalSuccessRate

| Evaluator | Value | Label | Explanation |
|---|---|---|---|
| `Builtin.Correctness` | N/A | ERR:LogEventMissingException | Session span data is incomplete. Span with ID: 90453e5cd76a35d1 and name: invoke_agent Strands Agents is missing a corre |
| `Builtin.GoalSuccessRate` | N/A | ERR:LogEventMissingException | Session span data is incomplete. Span with ID: 90453e5cd76a35d1 and name: invoke_agent Strands Agents is missing a corre |

### 5d. Multi-Turn: PTO Planning Session (3 turns) + Custom AssertionChecker

For multi-turn sessions, `EvaluationClient` fetches all spans for the session and evaluates
the complete conversation. The trajectory and assertions apply across all turns.

This scenario also exercises the custom `HRAssertionChecker` evaluator (SESSION level),
which uses `{actual_tool_trajectory}`, `{expected_tool_trajectory}`, and `{assertions}`
placeholders. A 3-turn session with distinct tool calls per turn gives the evaluator
a rich trajectory to compare against the expected sequence.

In [18]:
pto_planning_results = eval_client.run(
    evaluator_ids=[
        "Builtin.GoalSuccessRate",
        "Builtin.TrajectoryExactOrderMatch",
        "Builtin.TrajectoryInOrderMatch",
        "Builtin.TrajectoryAnyOrderMatch",
        "Builtin.Helpfulness",
        CUSTOM_ASSERTION_CHECKER_ID,   # SESSION (custom): {actual_tool_trajectory} + {expected_tool_trajectory} + {assertions}
    ],
    session_id=session_pto_planning,
    agent_id=AGENT_ID,
    look_back_time=timedelta(hours=2),
    reference_inputs=ReferenceInputs(
        expected_trajectory=["get_pto_balance", "submit_pto_request", "lookup_hr_policy"],
        assertions=[
            "Agent correctly reported 10 remaining PTO days for EMP-001 in turn 1",
            "Agent submitted a PTO request for December 23-25, 2026 in turn 2",
            "Agent correctly stated the 5-day PTO rollover limit in turn 3",
        ],
    ),
)

display_eval_results("PTO Planning — Multi-Turn (3 turns) + Custom AssertionChecker", pto_planning_results)

### PTO Planning — Multi-Turn (3 turns) + Custom AssertionChecker

| Evaluator | Value | Label | Explanation |
|---|---|---|---|
| `Builtin.GoalSuccessRate` | N/A | ERR:LogEventMissingException | Session span data is incomplete. Span with ID: 49b4e5ca7bfadb3b and name: invoke_agent Strands Agents is missing a corre |
| `Builtin.TrajectoryExactOrderMatch` | N/A | ERR:LogEventMissingException | Session span data is incomplete. Span with ID: 49b4e5ca7bfadb3b and name: invoke_agent Strands Agents is missing a corre |
| `Builtin.TrajectoryInOrderMatch` | N/A | ERR:LogEventMissingException | Session span data is incomplete. Span with ID: 49b4e5ca7bfadb3b and name: invoke_agent Strands Agents is missing a corre |
| `Builtin.TrajectoryAnyOrderMatch` | N/A | ERR:LogEventMissingException | Session span data is incomplete. Span with ID: 49b4e5ca7bfadb3b and name: invoke_agent Strands Agents is missing a corre |
| `Builtin.Helpfulness` | N/A | ERR:LogEventMissingException | Session span data is incomplete. Span with ID: 49b4e5ca7bfadb3b and name: invoke_agent Strands Agents is missing a corre |
| `Builtin.Helpfulness` | N/A | ERR:LogEventMissingException | Session span data is incomplete. Span with ID: 49b4e5ca7bfadb3b and name: invoke_agent Strands Agents is missing a corre |
| `Builtin.Helpfulness` | N/A | ERR:LogEventMissingException | Session span data is incomplete. Span with ID: 49b4e5ca7bfadb3b and name: invoke_agent Strands Agents is missing a corre |
| `HRAssertionChecker_a007e092-zUFaMBBhck` | N/A | ERR:LogEventMissingException | Session span data is incomplete. Span with ID: 49b4e5ca7bfadb3b and name: invoke_agent Strands Agents is missing a corre |

### 5e. Multi-Turn: New Employee Onboarding (4 turns)

This scenario checks that the agent correctly identifies which tool to use for each type of question
(policy lookup vs. benefits lookup vs. PTO balance check) across a realistic onboarding conversation.

In [19]:
onboarding_results = eval_client.run(
    evaluator_ids=[
        "Builtin.GoalSuccessRate",
        "Builtin.TrajectoryAnyOrderMatch",
        "Builtin.TrajectoryExactOrderMatch",
    ],
    session_id=session_onboarding,
    agent_id=AGENT_ID,
    look_back_time=timedelta(hours=2),
    reference_inputs=ReferenceInputs(
        expected_trajectory=[
            "lookup_hr_policy",    # turn 1: remote work policy
            "lookup_hr_policy",    # turn 2: PTO policy
            "get_benefits_summary", # turn 3: life insurance
            "get_pto_balance",     # turn 4: EMP-042 balance
        ],
        assertions=[
            "Agent looked up the remote work policy in turn 1 and mentioned 3 days per week",
            "Agent looked up the PTO policy in turn 2 and mentioned 15 days for new employees",
            "Agent looked up life insurance benefits in turn 3 and mentioned 2x annual salary",
            "Agent called get_pto_balance for EMP-042 in turn 4 and reported 13 remaining days",
        ],
    ),
)

display_eval_results("New Employee Onboarding — Multi-Turn Session (4 turns)", onboarding_results)

### New Employee Onboarding — Multi-Turn Session (4 turns)

| Evaluator | Value | Label | Explanation |
|---|---|---|---|
| `Builtin.GoalSuccessRate` | N/A | ERR:LogEventMissingException | Session span data is incomplete. Span with ID: 9b7976374cd80134 and name: invoke_agent Strands Agents is missing a corre |
| `Builtin.TrajectoryAnyOrderMatch` | N/A | ERR:LogEventMissingException | Session span data is incomplete. Span with ID: 9b7976374cd80134 and name: invoke_agent Strands Agents is missing a corre |
| `Builtin.TrajectoryExactOrderMatch` | N/A | ERR:LogEventMissingException | Session span data is incomplete. Span with ID: 9b7976374cd80134 and name: invoke_agent Strands Agents is missing a corre |

## Step 6: OnDemandEvaluationDatasetRunner — Automated Dataset Evaluation

`OnDemandEvaluationDatasetRunner` is the right tool when you have a **test dataset** and want to:
1. Automatically invoke your agent for each scenario
2. Collect CloudWatch spans
3. Run evaluators against each scenario's results

This is ideal for regression testing, CI/CD pipelines, and batch evaluation against curated datasets.

### Dataset structure

A dataset consists of **scenarios**, each with one or more **turns**. Optional ground-truth fields:
- `Turn.expected_response` — per-turn expected answer
- `TurnByTurnScenario.expected_trajectory` — ordered list of tool names
- `TurnByTurnScenario.assertions` — session-level assertions

### How OnDemandEvaluationDatasetRunner works

```
For each scenario:
  1. Create a new session ID
  2. Call your agent_invoker function for each turn
  3. Wait for CloudWatch spans to appear (evaluation_delay_seconds)
  4. Submit spans + ground truth to the evaluation service
  5. Collect and return results
```

In [20]:
from bedrock_agentcore.evaluation import (
    AgentInvokerInput,
    AgentInvokerOutput,
    CloudWatchAgentSpanCollector,
    Dataset,
    EvaluationRunConfig,
    OnDemandEvaluationDatasetRunner,
    EvaluatorConfig,
    Turn,
    PredefinedScenario,
)

In [21]:
def agent_invoker(invoker_input: AgentInvokerInput) -> AgentInvokerOutput:
    """
    Called by OnDemandEvaluationDatasetRunner once per turn. Invoke the HR assistant
    and return the text response.

    AgentInvokerInput fields:
      - payload:    The turn input (str or dict) from the dataset.
      - session_id: Framework-managed session ID, stable across all turns
                    in a scenario. Pass it to your agent for conversation continuity.
    """
    payload = invoker_input.payload
    body = {"prompt": payload} if isinstance(payload, str) else payload

    resp = agentcore_client.invoke_agent_runtime(
        agentRuntimeArn=AGENT_ARN,
        qualifier="DEFAULT",
        runtimeSessionId=invoker_input.session_id,
        payload=json.dumps(body).encode("utf-8"),
    )

    raw = resp["response"].read().decode("utf-8")
    parts = []
    for line in raw.splitlines():
        if line.startswith("data: "):
            chunk = line[len("data: "):]
            try:
                chunk = json.loads(chunk)
            except Exception:
                pass
            parts.append(str(chunk))
    return AgentInvokerOutput(agent_output="".join(parts) if parts else raw)

### 6a. Define the Evaluation Dataset

We define scenarios inline. A mix of single-turn and multi-turn scenarios exercises
different aspects of the agent.

In [22]:
dataset = Dataset(
    scenarios=[
        # --- Single-turn: PTO balance ---
        PredefinedScenario(
            scenario_id="pto-balance-check",
            turns=[
                Turn(
                    input="What is the current PTO balance for employee EMP-001?",
                    expected_response="Employee EMP-001 has 10 remaining PTO days out of 15 total (5 days used).",
                )
            ],
            expected_trajectory=["get_pto_balance"],
            assertions=[
                "Agent called get_pto_balance with employee_id=EMP-001",
                "Agent reported 10 remaining PTO days",
            ],
        ),

        # --- Single-turn: HR policy lookup ---
        PredefinedScenario(
            scenario_id="pto-policy-lookup",
            turns=[
                Turn(
                    input="What is the company PTO policy?",
                    expected_response="Full-time employees accrue 15 days of PTO per year. Requests must be submitted at least 2 business days in advance. Up to 5 unused days roll over each year.",
                )
            ],
            expected_trajectory=["lookup_hr_policy"],
            assertions=[
                "Agent called lookup_hr_policy with topic=pto",
                "Agent mentioned the 15-day annual accrual for full-time employees",
                "Agent mentioned the 2 business day advance notice requirement",
            ],
        ),

        # --- Single-turn: 401k benefits ---
        PredefinedScenario(
            scenario_id="401k-info",
            turns=[
                Turn(
                    input="How does the 401k match work?",
                    expected_response="The company matches 100% of contributions up to 4% of salary, plus 50% on the next 2%, for a total effective match of up to 5%. The match vests over 3 years.",
                )
            ],
            expected_trajectory=["get_benefits_summary"],
            assertions=[
                "Agent called get_benefits_summary with benefit_type=401k",
                "Agent correctly described the 4% full match and 50% match on next 2%",
                "Agent mentioned the 3-year vesting schedule",
            ],
        ),

        # --- Single-turn: check balance then submit PTO ---
        PredefinedScenario(
            scenario_id="check-and-submit-pto",
            turns=[
                Turn(
                    input="Check the PTO balance for EMP-002, and if they have at least 2 days, submit a request for 2026-05-26 to 2026-05-27.",
                    expected_response="EMP-002 has 3 remaining PTO days. PTO request submitted and approved for 2026-05-26 to 2026-05-27.",
                )
            ],
            expected_trajectory=["get_pto_balance", "submit_pto_request"],
            assertions=[
                "Agent first called get_pto_balance for EMP-002",
                "Agent confirmed 3 remaining days is sufficient",
                "Agent then called submit_pto_request for the correct dates",
            ],
        ),

        # --- Multi-turn: benefits exploration ---
        PredefinedScenario(
            scenario_id="benefits-exploration",
            turns=[
                Turn(
                    input="Can you walk me through the health insurance options?",
                    expected_response="The company covers 90% of premiums for employee-only coverage. Three plans are available: Blue Shield PPO, Kaiser HMO, and HDHP with HSA.",
                ),
                Turn(
                    input="What about dental?",
                    expected_response="The dental plan covers 100% of preventive care, 80% of basic restorative care, and 50% of major work, with a $2,000 annual maximum.",
                ),
                Turn(
                    input="And how much does the company contribute to the 401k?",
                    expected_response="The company matches 100% up to 4% of salary, plus 50% on the next 2%, for a total effective match of up to 5%.",
                ),
            ],
            expected_trajectory=["get_benefits_summary", "get_benefits_summary", "get_benefits_summary"],
            assertions=[
                "Agent called get_benefits_summary three times across the conversation",
                "Agent correctly described health, dental, and 401k benefits in their respective turns",
                "Agent maintained conversational context across all three turns",
            ],
        ),
    ]
)

print(f"Dataset contains {len(dataset.scenarios)} scenarios.")

Dataset contains 5 scenarios.


### 6b. Configure and Run OnDemandEvaluationDatasetRunner

In [23]:
# Span collector: polls CloudWatch for OTel spans emitted by the agent
span_collector = CloudWatchAgentSpanCollector(
    log_group_name=CW_LOG_GROUP,
    region=REGION,
    max_wait_seconds=180,
    poll_interval_seconds=15,
)

# Evaluator level cache — built-ins + custom evaluators
EVALUATOR_LEVELS = {
    "Builtin.GoalSuccessRate":            "SESSION",
    "Builtin.TrajectoryExactOrderMatch":  "SESSION",
    "Builtin.TrajectoryInOrderMatch":     "SESSION",
    "Builtin.TrajectoryAnyOrderMatch":    "SESSION",
    "Builtin.Correctness":                "TRACE",
    "Builtin.Helpfulness":                "TRACE",
    "Builtin.ResponseRelevance":          "TRACE",
    "Builtin.Coherence":                  "TRACE",
    "Builtin.InstructionFollowing":       "TRACE",
}
# Custom evaluators (IDs are runtime values from Step 3b)
EVALUATOR_LEVELS[CUSTOM_RESPONSE_SIMILARITY_ID] = "TRACE"
EVALUATOR_LEVELS[CUSTOM_ASSERTION_CHECKER_ID]   = "SESSION"

# Evaluator configuration — mix of built-in and custom evaluators
config = EvaluationRunConfig(
    evaluator_config=EvaluatorConfig(
        evaluator_ids=[
            "Builtin.Correctness",               # TRACE — expected_response
            "Builtin.GoalSuccessRate",            # SESSION — assertions
            "Builtin.TrajectoryExactOrderMatch",  # SESSION — expected_trajectory
            "Builtin.TrajectoryInOrderMatch",     # SESSION — expected_trajectory
            "Builtin.TrajectoryAnyOrderMatch",    # SESSION — expected_trajectory
            "Builtin.Helpfulness",               # TRACE — no ground truth
            "Builtin.ResponseRelevance",         # TRACE — no ground truth
            CUSTOM_RESPONSE_SIMILARITY_ID,       # TRACE (custom) — {assistant_turn} + {expected_response}
            CUSTOM_ASSERTION_CHECKER_ID,         # SESSION (custom) — {actual_tool_trajectory} + {assertions}
        ]
    ),
    evaluation_delay_seconds=180,
    max_concurrent_scenarios=3,
)

runner = OnDemandEvaluationDatasetRunner(region=REGION)
runner._evaluator_level_cache.update(EVALUATOR_LEVELS)

print("OnDemandEvaluationDatasetRunner configured. Starting evaluation...")
print(f"  Scenarios : {len(dataset.scenarios)}")
print(f"  Evaluators: {len(config.evaluator_config.evaluator_ids)} "
      f"(7 built-in + 2 custom)")
print(f"  Delay     : {config.evaluation_delay_seconds}s (waiting for CloudWatch ingestion)")

OnDemandEvaluationDatasetRunner configured. Starting evaluation...
  Scenarios : 5
  Evaluators: 9 (7 built-in + 2 custom)
  Delay     : 180s (waiting for CloudWatch ingestion)


In [24]:
# Run the evaluation.
# OnDemandEvaluationDatasetRunner will:
#   1. Invoke agent_invoker for each turn in each scenario
#   2. Wait evaluation_delay_seconds for CloudWatch ingestion
#   3. Submit spans to the evaluation service
#   4. Return aggregated results

eval_result = runner.run(
    config=config,
    dataset=dataset,
    agent_invoker=agent_invoker,
    span_collector=span_collector,
)

completed = sum(1 for sr in eval_result.scenario_results if sr.status == "COMPLETED")
failed = sum(1 for sr in eval_result.scenario_results if sr.status == "FAILED")
print(f"\nEvaluation complete: {completed} completed, {failed} failed out of {len(eval_result.scenario_results)} scenarios.")


Evaluation complete: 5 completed, 0 failed out of 5 scenarios.


### 6c. Inspect Results

In [25]:
def display_runner_results(eval_result) -> None:
    """Display OnDemandEvaluationDatasetRunner results as a markdown table per scenario."""
    for sr in eval_result.scenario_results:
        if sr.status == "FAILED":
            display(Markdown(f"**Scenario `{sr.scenario_id}`** — FAILED: {sr.error}"))
            continue

        rows = ["| Evaluator | Value | Label | Explanation |",
                "|---|---|---|---|"]
        for er in sr.evaluator_results:
            for res in er.results:
                value = str(res.get("value", res.get("score", "N/A")))
                lbl = str(res.get("label", res.get("rating", "")))
                explanation = (res.get("explanation", "") or "")[:130].replace("\n", " ")
                error_code = res.get("errorCode")
                if error_code:
                    lbl = f"ERR:{error_code}"
                    explanation = (res.get("errorMessage", "") or "")[:130]
                rows.append(f"| `{er.evaluator_id[:40]}` | {value} | {lbl} | {explanation} |")

        md = f"### Scenario: `{sr.scenario_id}`\n\n" + "\n".join(rows)
        display(Markdown(md))


display_runner_results(eval_result)

### Scenario: `pto-balance-check`

| Evaluator | Value | Label | Explanation |
|---|---|---|---|
| `Builtin.Correctness` | 1.0 | Correct | The agent response provides all the core factual information present in the expected response: employee EMP-001 has 15 total PTO d |
| `Builtin.GoalSuccessRate` | 1.0 | Yes | The conversation record shows that the agent called the 'get_pto_balance' tool with the parameter 'employee_id': 'EMP-001', which  |
| `Builtin.TrajectoryExactOrderMatch` | 1.0 | Yes | Exact match: Actual trajectory ['get_pto_balance'] matches expected trajectory ['get_pto_balance'] |
| `Builtin.TrajectoryInOrderMatch` | 1.0 | Yes | In-order match: All expected tools ['get_pto_balance'] found in order within actual ['get_pto_balance'] |
| `Builtin.TrajectoryAnyOrderMatch` | 1.0 | Yes | Any-order match: All expected tools ['get_pto_balance'] found in actual ['get_pto_balance'] |
| `Builtin.Helpfulness` | 0.83 | Very Helpful | The user's goal is clear and straightforward: to obtain the current PTO balance for employee EMP-001. The assistant successfully r |
| `Builtin.ResponseRelevance` | 1.0 | Completely Yes | The user asked for the current PTO balance for employee EMP-001. The LLM response directly addresses this question by providing th |
| `HRResponseSimilarity_a007e092-v0ojZ8ARHR` | 1.0 | highly_similar | The agent's response includes all key facts present in the expected response: the total number of PTO days (15), the number of use |
| `HRAssertionChecker_a007e092-zUFaMBBhck` | 1.0 | passed | The agent followed the expected tool trajectory by calling 'get_pto_balance'. Additionally, the agent satisfied the assertion that |

### Scenario: `pto-policy-lookup`

| Evaluator | Value | Label | Explanation |
|---|---|---|---|
| `Builtin.Correctness` | 1.0 | Correct | The agent response includes all the core factual information from the expected response: (1) full-time employees accrue 15 days of |
| `Builtin.GoalSuccessRate` | 1.0 | Yes | Evaluating the three success assertions:  1. **Agent called lookup_hr_policy with topic=pto**: The tool execution history shows th |
| `Builtin.TrajectoryExactOrderMatch` | 1.0 | Yes | Exact match: Actual trajectory ['lookup_hr_policy'] matches expected trajectory ['lookup_hr_policy'] |
| `Builtin.TrajectoryInOrderMatch` | 1.0 | Yes | In-order match: All expected tools ['lookup_hr_policy'] found in order within actual ['lookup_hr_policy'] |
| `Builtin.TrajectoryAnyOrderMatch` | 1.0 | Yes | Any-order match: All expected tools ['lookup_hr_policy'] found in actual ['lookup_hr_policy'] |
| `Builtin.Helpfulness` | 0.83 | Very Helpful | The user's goal is clear and straightforward: to learn about the company's PTO policy. The assistant's response directly addresses |
| `Builtin.ResponseRelevance` | 1.0 | Completely Yes | The user asked 'What is the company PTO policy?' The LLM response directly addresses this question by providing the complete PTO p |
| `HRResponseSimilarity_a007e092-v0ojZ8ARHR` | 0.5 | partially_similar | The agent's response includes all the key facts from the expected response: full-time employees accrue 15 days of PTO per year, re |
| `HRAssertionChecker_a007e092-zUFaMBBhck` | 1.0 | passed | The agent followed the expected tool trajectory by calling 'lookup_hr_policy' with the topic 'pto'. Additionally, the agent mentio |

### Scenario: `401k-info`

| Evaluator | Value | Label | Explanation |
|---|---|---|---|
| `Builtin.Correctness` | 1.0 | Correct | The agent response accurately conveys all the core factual information from the expected response: (1) 100% match up to 4% of sala |
| `Builtin.GoalSuccessRate` | 1.0 | Yes | Evaluating the three success assertions:  1. "Agent called get_benefits_summary with benefit_type=401k" - The tool execution histo |
| `Builtin.TrajectoryExactOrderMatch` | 1.0 | Yes | Exact match: Actual trajectory ['get_benefits_summary'] matches expected trajectory ['get_benefits_summary'] |
| `Builtin.TrajectoryInOrderMatch` | 1.0 | Yes | In-order match: All expected tools ['get_benefits_summary'] found in order within actual ['get_benefits_summary'] |
| `Builtin.TrajectoryAnyOrderMatch` | 1.0 | Yes | Any-order match: All expected tools ['get_benefits_summary'] found in actual ['get_benefits_summary'] |
| `Builtin.Helpfulness` | 0.83 | Very Helpful | The user's goal is clearly to understand how the 401k match works at their company. The assistant's response directly addresses th |
| `Builtin.ResponseRelevance` | 1.0 | Completely Yes | The user asked 'How does the 401k match work?' The LLM response directly addresses this question by explaining the 401k matching s |
| `HRResponseSimilarity_a007e092-v0ojZ8ARHR` | 1.0 | highly_similar | The agent's response includes all key facts from the expected response, such as the company's matching contribution of 100% up to  |
| `HRAssertionChecker_a007e092-zUFaMBBhck` | 1.0 | passed | The agent followed the expected tool trajectory by calling 'get_benefits_summary'. It also satisfied all assertions by correctly d |

### Scenario: `check-and-submit-pto`

| Evaluator | Value | Label | Explanation |
|---|---|---|---|
| `Builtin.Correctness` | 1.0 | Correct | The agent response contains all the core factual information from the expected response: (1) EMP-002 has 3 remaining PTO days, (2) |
| `Builtin.GoalSuccessRate` | 1.0 | Yes | The agent successfully completed all three assertions: (1) First called get_pto_balance for EMP-002, which returned 3 remaining da |
| `Builtin.TrajectoryExactOrderMatch` | 1.0 | Yes | Exact match: Actual trajectory ['get_pto_balance', 'submit_pto_request'] matches expected trajectory ['get_pto_balance', 'submit_p |
| `Builtin.TrajectoryInOrderMatch` | 1.0 | Yes | In-order match: All expected tools ['get_pto_balance', 'submit_pto_request'] found in order within actual ['get_pto_balance', 'sub |
| `Builtin.TrajectoryAnyOrderMatch` | 1.0 | Yes | Any-order match: All expected tools ['get_pto_balance', 'submit_pto_request'] found in actual ['get_pto_balance', 'submit_pto_requ |
| `Builtin.Helpfulness` | 0.83 | Very Helpful | The user's goal was clear: check if EMP-002 has at least 2 days of PTO balance, and if so, submit a request for the specified date |
| `Builtin.ResponseRelevance` | 1.0 | Completely Yes | The user's question asks to: (1) check the PTO balance for EMP-002, and (2) if they have at least 2 days, submit a request for 202 |
| `HRResponseSimilarity_a007e092-v0ojZ8ARHR` | 0.5 | partially_similar | The agent's response includes all the key information from the expected response: EMP-002 has 3 remaining PTO days, and the PTO re |
| `HRAssertionChecker_a007e092-zUFaMBBhck` | N/A | ERR:ValueError | No score found in evaluation result |

### Scenario: `benefits-exploration`

| Evaluator | Value | Label | Explanation |
|---|---|---|---|
| `Builtin.Correctness` | 1.0 | Correct | The agent response contains all the core factual information from the expected response: (1) the company covers 90% of premiums fo |
| `Builtin.Correctness` | 1.0 | Correct | The agent response contains all the core factual information from the expected response: 100% coverage for preventive care, 80% fo |
| `Builtin.Correctness` | 1.0 | Correct | The agent response correctly conveys the core factual information about the company's 401k contribution. It states that the compan |
| `Builtin.GoalSuccessRate` | 1.0 | Yes | Evaluating the three success assertions:  1. **Agent called get_benefits_summary three times across the conversation**: The tool e |
| `Builtin.TrajectoryExactOrderMatch` | 1.0 | Yes | Exact match: Actual trajectory ['get_benefits_summary', 'get_benefits_summary', 'get_benefits_summary'] matches expected trajector |
| `Builtin.TrajectoryInOrderMatch` | 1.0 | Yes | In-order match: All expected tools ['get_benefits_summary', 'get_benefits_summary', 'get_benefits_summary'] found in order within  |
| `Builtin.TrajectoryAnyOrderMatch` | 1.0 | Yes | Any-order match: All expected tools ['get_benefits_summary', 'get_benefits_summary', 'get_benefits_summary'] found in actual ['get |
| `Builtin.Helpfulness` | 0.83 | Very Helpful | The user's goal is to understand the health insurance options available to them. The assistant's response directly addresses this  |
| `Builtin.Helpfulness` | 0.83 | Very Helpful | The user's goal is to understand health insurance options at Acme Corp, and they've now expanded their inquiry to include dental b |
| `Builtin.Helpfulness` | 0.83 | Very Helpful | The user's goal is to understand employee benefits at Acme Corp, specifically asking about 401k company contributions. The assista |
| `Builtin.ResponseRelevance` | 1.0 | Completely Yes | The user asked to be walked through the health insurance options. The LLM response directly addresses this question by providing a |
| `Builtin.ResponseRelevance` | 1.0 | Completely Yes | The user's question is 'What about dental?' which is a follow-up question asking about dental insurance options. The LLM response  |
| `Builtin.ResponseRelevance` | 1.0 | Completely Yes | The user's question is: 'And how much does the company contribute to the 401k?' The LLM response provides information about the 40 |
| `HRResponseSimilarity_a007e092-v0ojZ8ARHR` | 1.0 | highly_similar | The agent's response includes all the key facts mentioned in the expected response. It accurately states the premium coverage perc |
| `HRResponseSimilarity_a007e092-v0ojZ8ARHR` | 1.0 | highly_similar | The agent's response provides a comprehensive summary of the dental insurance options, including coverage details, annual maximum  |
| `HRResponseSimilarity_a007e092-v0ojZ8ARHR` | 1.0 | highly_similar | The agent's response provides a detailed summary that includes all the key facts mentioned in the expected response. It correctly  |
| `HRAssertionChecker_a007e092-zUFaMBBhck` | 1.0 | passed | The agent followed the expected tool trajectory exactly as specified, calling 'get_benefits_summary' three times. Additionally, th |

In [26]:
# Aggregate summary: average score per evaluator across all scenarios
from collections import defaultdict

scores_by_evaluator = defaultdict(list)
for sr in eval_result.scenario_results:
    if sr.status != "COMPLETED":
        continue
    for er in sr.evaluator_results:
        for res in er.results:
            if "value" in res and res["value"] is not None and not res.get("errorCode"):
                scores_by_evaluator[er.evaluator_id].append(float(res["value"]))

print("\nEvaluator Summary (average score across all scenarios)")
print("=" * 60)
for evaluator_id, scores in sorted(scores_by_evaluator.items()):
    avg = sum(scores) / len(scores)
    print(f"  {evaluator_id:<45} avg={avg:.2f}  (n={len(scores)})")


Evaluator Summary (average score across all scenarios)
  Builtin.Correctness                           avg=1.00  (n=7)
  Builtin.GoalSuccessRate                       avg=1.00  (n=5)
  Builtin.Helpfulness                           avg=0.83  (n=7)
  Builtin.ResponseRelevance                     avg=1.00  (n=7)
  Builtin.TrajectoryAnyOrderMatch               avg=1.00  (n=5)
  Builtin.TrajectoryExactOrderMatch             avg=1.00  (n=5)
  Builtin.TrajectoryInOrderMatch                avg=1.00  (n=5)
  HRAssertionChecker_a007e092-zUFaMBBhck        avg=1.00  (n=4)
  HRResponseSimilarity_a007e092-v0ojZ8ARHR      avg=0.86  (n=7)


### 6d. Save Results to File

In [27]:
import os
from datetime import datetime

os.makedirs("results", exist_ok=True)
timestamp = datetime.utcnow().strftime("%Y%m%d_%H%M%S")
results_path = f"results/groundtruth_eval_{timestamp}.json"

with open(results_path, "w") as f:
    json.dump(eval_result.model_dump(), f, indent=2, default=str)

print(f"Results saved to: {results_path}")

Results saved to: results/groundtruth_eval_20260331_190711.json


/var/folders/9h/jk764rms7493dnsp6_d37wqm0000gq/T/ipykernel_96692/3226257259.py:5: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.utcnow().strftime("%Y%m%d_%H%M%S")


## Step 7: Cleanup

Delete the agent runtime endpoint when you're done to avoid ongoing costs.

In [28]:
# Uncomment to delete the agent runtime
# agent_runtime.delete()
# print("Agent runtime deleted.")

print("Cleanup skipped. Uncomment the cell above to delete the agent runtime.")

Cleanup skipped. Uncomment the cell above to delete the agent runtime.


### Key takeaways

| | EvaluationClient | OnDemandEvaluationDatasetRunner |
|---|---|---|
| **When to use** | You have existing sessions | You have a test dataset |
| **Agent invocation** | Not included | Automatic |
| **Best for** | Post-hoc analysis, debugging | Regression testing, CI/CD |
| **Input** | session_id | Dataset of scenarios |

### Built-in evaluator reference

| Evaluator | Level | Ground truth required |
|---|---|---|
| `Builtin.Correctness` | TRACE | `expected_response` |
| `Builtin.GoalSuccessRate` | SESSION | `assertions` |
| `Builtin.TrajectoryExactOrderMatch` | SESSION | `expected_trajectory` |
| `Builtin.TrajectoryInOrderMatch` | SESSION | `expected_trajectory` |
| `Builtin.TrajectoryAnyOrderMatch` | SESSION | `expected_trajectory` |
| `Builtin.Helpfulness` | TRACE | None |
| `Builtin.ResponseRelevance` | TRACE | None |
| `Builtin.Coherence` | TRACE | None |

### Custom evaluator ground truth placeholders

Custom (LLM-as-a-judge) evaluators reference ground truth via placeholders in their `instructions`.

| Level | Placeholder | Filled from |
|---|---|---|
| TRACE | `{assistant_turn}` | Agent's actual response |
| TRACE | `{expected_response}` | `ReferenceInputs.expected_response` |
| TRACE | `{context}` | Session context |
| SESSION | `{actual_tool_trajectory}` | Tools called by the agent |
| SESSION | `{expected_tool_trajectory}` | `ReferenceInputs.expected_trajectory` |
| SESSION | `{assertions}` | `ReferenceInputs.assertions` |
| SESSION | `{available_tools}` | Tools available to the agent |